# VGG16 — Very Deep Convolutional Networks (TensorFlow / Keras)
**Paper:** Very Deep Convolutional Networks for Large-Scale Image Recognition (Simonyan & Zisserman, ICLR 2015)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Conv blocks | 5 (2-2-3-3-3 layers per block) |
| FC head | Dense(4096) × 2 + Dense(classes) |
| Flatten dim | 7×7×512 = 25,088 |
| Parameters | ~138M |
| Top-1 (ImageNet) | ~71.3% |
| Input size | 224×224 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — VGG16 Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def build_vgg16(num_classes=1000, input_shape=(224, 224, 3)):
    """
    VGG16 — Very Deep Convolutional Networks for Large-Scale Image Recognition.
    Paper: Simonyan & Zisserman, ICLR 2015.

    Architecture:
      Block 1 : Conv(64)  x2  + MaxPool  ->  64 x 112x112
      Block 2 : Conv(128) x2  + MaxPool  -> 128 x  56x56
      Block 3 : Conv(256) x3  + MaxPool  -> 256 x  28x28
      Block 4 : Conv(512) x3  + MaxPool  -> 512 x  14x14
      Block 5 : Conv(512) x3  + MaxPool  -> 512 x   7x7
      Head    : Flatten -> Dense(4096) -> Dense(4096) -> Dense(num_classes)

    All convolutions: 3x3, padding='same', ReLU.
    All max-pools   : 2x2, stride=2.
    """
    inputs = keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='block1_conv1')(inputs)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='block1_conv2')(x)
    x = layers.MaxPooling2D(2, strides=2, name='block1_pool')(x)   # 112x112

    # Block 2
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='block2_conv1')(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='block2_conv2')(x)
    x = layers.MaxPooling2D(2, strides=2, name='block2_pool')(x)   # 56x56

    # Block 3
    x = layers.Conv2D(256, 3, padding='same', activation='relu', name='block3_conv1')(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu', name='block3_conv2')(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu', name='block3_conv3')(x)
    x = layers.MaxPooling2D(2, strides=2, name='block3_pool')(x)   # 28x28

    # Block 4
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block4_conv1')(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block4_conv2')(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block4_conv3')(x)
    x = layers.MaxPooling2D(2, strides=2, name='block4_pool')(x)   # 14x14

    # Block 5
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block5_conv1')(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block5_conv2')(x)
    x = layers.Conv2D(512, 3, padding='same', activation='relu', name='block5_conv3')(x)
    x = layers.MaxPooling2D(2, strides=2, name='block5_pool')(x)   # 7x7

    # Classifier head
    x = layers.Flatten(name='flatten')(x)                          # 25088
    x = layers.Dense(4096, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(4096, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='vgg16')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_vgg16(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=100)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    shear_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'vgg16_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)
print('Best model saved: vgg16_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VGG16 — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
from PIL import Image

def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    labels    = [CLASS_NAMES[i] for i in top_idx]
    top_probs = [probs[i] * 100 for i in top_idx]
    axes[1].barh(labels[::-1], top_probs[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'VGG16 — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')